## Check Environment

In [ ]:
!nvidia-smi
!nvcc --version

## Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/00 GEMM_project')
!ls src/ | grep attention

## Compile Fused Attention Kernel

In [ ]:
!nvcc -O3 -arch=sm_80 src/08_fused_attention.cu -o 08_fused_attention
!echo 'Compile done'

## Benchmark vs PyTorch Baseline

In [ ]:
!pip install torch -q

import torch
import torch.nn.functional as F
import math

def attention_naive(Q, K, V):
    d = Q.shape[-1]
    S = Q @ K.transpose(-2, -1) / math.sqrt(d)
    P = torch.softmax(S, dim=-1)
    return P @ V

def benchmark(fn, *args, runs=50, warmup=5, label=''):
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(runs):
        fn(*args)
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end) / runs
    B, H, N, d = args[0].shape
    tflops = (4.0 * B * H * N * N * d) / (ms * 1e-3) / 1e12
    print(f'{label:30s}  {ms:.3f} ms  {tflops:.2f} TFLOPS')
    return ms, tflops

device = torch.device('cuda')
dtype  = torch.float32

print(f'\n{"Config":30s}  {"Time":>10}  {"TFLOPS":>10}')
print('-' * 56)

for N in [1024, 2048, 4096]:
    B, H, d = 1, 8, 64
    Q = torch.randn(B, H, N, d, device=device, dtype=dtype)
    K = torch.randn(B, H, N, d, device=device, dtype=dtype)
    V = torch.randn(B, H, N, d, device=device, dtype=dtype)
    print(f'\nN={N}')
    benchmark(attention_naive, Q, K, V, label='  pytorch naive')
    benchmark(F.scaled_dot_product_attention, Q, K, V, label='  pytorch SDPA')

In [ ]:
# Run our CUDA kernel for comparison
print('Our fused attention kernel:')
!./08_fused_attention 1024 8
!./08_fused_attention 2048 8
!./08_fused_attention 4096 8

## NCU Profiling — Roofline

In [ ]:
# Roofline profile (~5 passes, fast, timing is valid)
!ncu --force-overwrite \
    --set roofline \
    --kernel-name fused_attention \
    -o results/08_fused_attention_roofline \
    ./08_fused_attention 4096 8

!echo 'Roofline profile saved to results/08_fused_attention_roofline.ncu-rep'

## NCU Profiling — Full (stall analysis, timing NOT meaningful)

In [ ]:
# Full profile (~49 passes) — only use for stall/memory workload analysis
# Wall-clock time will be inflated ~50x, ignore throughput numbers here
!ncu --force-overwrite \
    --set full \
    --kernel-name fused_attention \
    -o results/08_fused_attention_full \
    ./08_fused_attention 4096 8

!echo 'Full profile saved to results/08_fused_attention_full.ncu-rep'

## NCU Profiling — GEMM Reference (BK16_pad2_lb2)

For HBM traffic comparison: attention kernel vs best GEMM kernel.

In [ ]:
# Compile and profile the best GEMM tuning variant for comparison
!nvcc -O3 -arch=sm_80 -lcublas tuning/BK16_pad2_lb2/kernel.cu -o tuning/BK16_pad2_lb2/kernel

!ncu --force-overwrite \
    --set roofline \
    --kernel-name gemm_final \
    -o results/tuning_BK16_pad2_lb2_roofline \
    ./tuning/BK16_pad2_lb2/kernel 4096

!echo 'GEMM profile saved to results/tuning_BK16_pad2_lb2_roofline.ncu-rep'

## List Results

In [ ]:
!ls -lh results/*.ncu-rep